## 2. Labelling Transcripts

__Conversations at Scale: Robust AI-led Interviews (https://ssrn.com/abstract=4974382)__

This notebook `02-labelling.ipynb` discusses how to label transcripts, thereby transforming qualitative into quantitative data. The scale of transcripts from AI-based interviews can make it particularly useful to automate parts of this process. In detail, each transcript is passed into the language model's context, and the language model is prompted to label whether a certain topic is present in the transcript or not, as well as to give an explanation of its decisions. We used such an approach e.g. in the study on meaning to label whether certain themes (such as "family and friends", "work", etc.) were mentioned in each interview transcript. Please refer to the paper for more details on the approach and the correlation of AI and human topic labelling.

Before running this notebook, please follow the README.md file on how to set up VS Code for running Jupyter notebooks, as well as creating the `interviews` conda environment with all necessary libraries. Connect to this environment in VS Code by clicking on "Select Kernel" in the top right corner and selecting `interviews`. Afterwards, open the file `details.py` and paste your API keys. __Ensure that the file `details.py` containing your API keys is not accidentally shared in a public repository.__ So far the code supports the OpenAI, Anthropic, Google, and Microsoft Azure APIs. The underlying functions in `core.py` can be extended relatively easily to support other language model APIs as well, or, if sufficient local compute is available, the code could be reworked to run with local language models.

__Note:__ In the following examples, reasoning efforts are disabled or set to low to allow for faster responses and lower costs, but you may find more accurate responses if you set them to high (assuming a given language model supports reasoning).

To start, run the cells below step-by-step.

In [ ]:
import core
import details

First, set the `transcript_directory` variable to the directory where the interview transcripts are stored. The code below works with the transcript text files which the interview platform saves at the end of interviews.

In [ ]:
# Load transcripts
transcript_directory = "..."
transcript_paths = core.load_file_paths(transcript_directory)

Next, adjust the prompt by replacing `...` with a description of the topic you would like to label or detect in the transcripts. In the meaning study of the paper, this could e.g. be `topic = "Creative Pursuits (e.g., art, music, writing) as a major source of meaning in their life"`.

In [ ]:
# Prompt
topic = "..."

prompt = f"""In the following interview, does the respondent mention {topic}?

Please reason in one sentence before you assign a label of 0 (no) or 1 (yes), responding in JSON format: '{{"reasoning": "<your reasoning>", "label": 0 or 1}}'"""

system = "You are carefully analyzing the interview transcript."

# Print prompt for verification
print(prompt)

The next code cells describe how to run labelling with different APIs and language models. For additional details on the input parameters of the `label_texts` function, see the docstring and code in the module `core.py`.

<br>

__OpenAI API__

The reasoning effort can also be set to values such as `low`, `medium`, or `high` depending on the model used. For other API arguments which can be set in `additional_api_kwargs`, see the documentation of the API https://platform.openai.com/docs/api-reference/introduction.

In [ ]:
from openai import OpenAI

labels_gpt = core.label_texts(
    transcript_paths,
    prompt=prompt,
    system=system,
    model="gpt-5.2-2025-12-11",
    additional_api_kwargs={"reasoning": {"effort": "none"}},
    client=OpenAI(api_key=details.KEY_OPENAI),
)
labels_gpt

__Anthropic API__

Thinking mode can be `enabled`, but requires setting e.g. {"budget_tokens": 10000} to specify the budget for thinking tokens generated by the model; for other API arguments which can be set in `additional_api_kwargs`, see the documentation of the API https://docs.claude.com/en/api/overview.

In [ ]:
import anthropic

labels_claude = core.label_texts(
    transcript_paths,
    prompt=prompt,
    system=system,
    model="claude-sonnet-4-5-20250929",
    additional_api_kwargs={
        "max_tokens": 500,
        "temperature": 0.0,
        "thinking": {"type": "disabled"},
    },
    client=anthropic.Anthropic(api_key=details.KEY_ANTHROPIC),
)
labels_claude

__Google API__

The thinking level can be set to values such as `minimal`, `low`, `medium`, or `high` depending on the model used. For other API arguments which can be set in `additional_api_kwargs`, see the documentation of the API https://ai.google.dev/gemini-api/docs.

In [ ]:
from google import genai

labels_gemini = core.label_texts(
    transcript_paths,
    prompt=prompt,
    system=system,
    model="gemini-3-flash-preview",
    additional_api_kwargs={
        "config": {
            "max_output_tokens": 500,
            "thinking_config": {"thinking_level": "minimal"},
        }
    },
    client=genai.Client(api_key=details.KEY_GOOGLE),
)
labels_gemini

__Microsoft Azure Inference API__

This API is one example of many APIs which offer access to a range of open-source models. A new resource can be created at https://ai.azure.com and language models such as e.g. Llama or DeepSeek be selected. Beyond the key, the endpoint and version have to be added to `details.py` as well. For API arguments which can be set in `additional_api_kwargs`, see the API documentation https://learn.microsoft.com/en-us/rest/api/aifoundry/modelinference/.

In [ ]:
from azure.ai.inference import ChatCompletionsClient
from azure.core.credentials import AzureKeyCredential

labels_llama = core.label_texts(
    transcript_paths,
    prompt=prompt,
    system=system,
    model="Llama-3.3-70B-Instruct",
    additional_api_kwargs={"max_tokens": 500, "temperature": 0.0},
    client=ChatCompletionsClient(
        endpoint=details.ENDPOINT_AZURE,
        credential=AzureKeyCredential(details.KEY_AZURE),
        api_version=details.VERSION_AZURE,
    ),
)
labels_llama